In [1]:
!pwd

/raid/home/fdivaler/XAI_CL/CIPNet


In [2]:
%cd ./src

/raid/home/fdivaler/XAI_CL/CIPNet/src


/raid/home/fdivaler/miniconda3/envs/PIPNet/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [107]:
import random
import numpy as np
import os
import torch
import torch.nn.functional as F
import torch.nn.functional as F
from PIL import ImageDraw as D
import torchvision.transforms as transforms
from networks.cipnet_utils.util.data import TrivialAugmentWideNoColor, TrivialAugmentWideNoShape
from datasets.data_loader import get_cub_datasets, get_transforms, dataset_config
# from datasets.data_loader.dataset_config import dataset_config
from torch.utils.data.distributed import DistributedSampler
from torch.utils import data
import datasets.base_dataset as basedat

In [108]:
#Set seed
cudnn_deterministic = True
def seed_everything(seed=0):
    """Fix all random seeds"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = cudnn_deterministic
seed=1
seed_everything(seed)

In [ ]:
#Set device
device = 'cpu'
#SET PATH FOR THE EXPERIMENT
path = "/raid/home/fdivaler/XAI_CL/ICICLE/frozenheads/cub_200_2011_cropped_icicle_4tasks/seed1_savemodel_sharedTau100.0_decorrelationHeads0.005_LASimSiamTrick_CosineLinear16.0InitNormal_protoreg8.0_completeSoftmax/"
#Load checkpoint
checkpoint = torch.load(path+"models/model_task3.pt", map_location=device)


In [113]:
#Construct CIPNet
from networks.cipnet_utils.cipnet import construct_CIPNet
from wrapperDDP import wrapper
num_classes = 200 #196
num_tasks = 4
model = construct_CIPNet(num_classes//num_tasks, "convnext_tiny_26",True, False)
model = wrapper(model, devices = device, parallelization=None)
for i in range(num_tasks-1):
    model.module.add_head(bias=False)

Using CONVNEXT_TINY_26 as base architecture
Number of prototypes:  768
NUM PROTOTYPES 768
CIPNET CLS HEAD ADDED AND INITIALIZED
CIPNET CLS HEAD ADDED AND INITIALIZED
CIPNET CLS HEAD ADDED AND INITIALIZED


In [114]:
#Load model
model.module._net.load_state_dict(checkpoint['backbone'])
model.module._classification.load_state_dict(checkpoint['classifiers'])
if 'tau' in checkpoint.keys():
    model.module.tau = checkpoint['tau']

proto_used = checkpoint['proto_used']
for k,v in proto_used.items():
    proto_used[k] = v.cpu()
proto_idxs = checkpoint['proto_idxs'].cpu()

model.to(device)
model.eval()

print("MODELLO CARICATO")

MODELLO CARICATO


In [ ]:
#Loader functions
def get_cub_datasetss(dataset, path, num_tasks, nc_first_task, validation, trn_transform, tst_transform, transform1=None, transform2=None, transform_noaug=None, class_order=None, repeat_task_0=False):
    """Extract datasets and create Dataset class"""
    # print("nc_first_task",nc_first_task, flush=True)
    # print("class_order",class_order, flush=True)
    trn_dset, val_dset, tst_dset, psh_dset = [], [], [], []

    # read data paths and compute splits -- path needs to have a train.txt and a test.txt with image-label pairs
    all_data, taskcla, class_indices, idxs = basedat.get_data2(path, num_tasks=num_tasks, nc_first_task=nc_first_task,
                                                        validation=validation, shuffle_classes=class_order is None,
                                                        class_order=class_order)
    # set dataset type
    Dataset=basedat.BaseDatasetCIPNet

    if repeat_task_0:
        taskcla.insert(0, taskcla[0])
    # get datasets, apply correct label offsets for each task
    offset = 0
    for task in range(num_tasks):
        all_data[task]['trn']['y'] = [label + offset for label in all_data[task]['trn']['y']]
        all_data[task]['val']['y'] = [label + offset for label in all_data[task]['val']['y']]
        all_data[task]['psh']['y'] = [label + offset for label in all_data[task]['psh']['y']]
        all_data[task]['tst']['y'] = [label + offset for label in all_data[task]['tst']['y']]
        idxs[task]['tst'] = {k+offset:v for k,v in idxs[task]['tst'].items()}
        idxs[task]['trn'] = {k+offset:v for k,v in idxs[task]['trn'].items()}
        trn_dset.append(Dataset(all_data[task]['trn'], trn_transform, class_indices, name=dataset,
                                transform1=transform1, transform2=transform2, transform_noaug=transform_noaug, test=True))####################
        val_dset.append(Dataset(all_data[task]['val'], tst_transform, class_indices, name=dataset,
                                transform1=transform1, transform2=transform2, transform_noaug=transform_noaug, test=True))
        tst_dset.append(Dataset(all_data[task]['tst'], tst_transform, class_indices, name=dataset,
                                transform1=transform1, transform2=transform2, transform_noaug=transform_noaug, test=True))
        psh_dset.append(Dataset(all_data[task]['psh'], tst_transform, class_indices, name=dataset,
                                transform1=transform1, transform2=transform2))
        
        offset += taskcla[task][1]
    if repeat_task_0:
        # print("SONO ENTRATO IN REPEAT TASK 0")
        trn_dset.insert(0, Dataset(all_data[0]['trn'], trn_transform, class_indices, name=dataset,
                                    transform1=transform1, transform2=transform2))
        val_dset.insert(0, Dataset(all_data[0]['val'], trn_transform, class_indices, name=dataset,
                                    transform1=transform1, transform2=transform2, transform_noaug=transform_noaug, test=True))
        tst_dset.insert(0, Dataset(all_data[0]['tst'], trn_transform, class_indices, name=dataset,
                                    transform1=transform1, transform2=transform2, transform_noaug=transform_noaug, test=True))
        psh_dset.insert(0, Dataset(all_data[0]['psh'], trn_transform, class_indices, name=dataset,
                                    transform1=transform1, transform2=transform2))
        
    return trn_dset, val_dset, tst_dset, psh_dset, taskcla, idxs

def get_loaderss(datasets, num_tasks, nc_first_task, batch_size, num_workers, pin_memory, validation=.1,
                repeat_task_0=False, parallelization="DP"):
    """Apply transformations to Datasets and create the DataLoaders for each task"""

    trn_load, val_load, tst_load, psh_load = [], [], [], []
    taskcla = []
    dataset_offset = 0
    for idx_dataset, cur_dataset in enumerate(datasets, 0):
        print(f"Processing dataset: {cur_dataset}")
        # get configuration for current dataset
        dc = dataset_config[cur_dataset]
        print(dc)
        # transformations
        trn_transform, tst_transform = get_transforms(resize=dc['resize'],
                                                      pad=dc['pad'],
                                                      crop=dc['crop'],
                                                      flip=dc['flip'],
                                                      normalize=dc['normalize'],
                                                      extend_channel=dc['extend_channel'],
                                                      online_augment=dc['online_augment'],)
        # datasets
        if 'cub_200_2011' in cur_dataset or 'cars' in cur_dataset:
            img_size = 224 # SPERO SIA 224 SEMPRE
            shape = (3, img_size, img_size)
            mean = (0.485, 0.456, 0.406)
            std = (0.229, 0.224, 0.225)
            normalize = transforms.Normalize(mean=mean,std=std)
            
            transform_noaug = transforms.Compose([
                                    transforms.Resize(size=(img_size, img_size)),
                                    transforms.ToTensor(),
                                    normalize
                                ])
            # transform1p = None
            
            transform1 = transforms.Compose([
                transforms.Resize(size=(img_size+8, img_size+8)), 
                TrivialAugmentWideNoColor(),
                transforms.RandomHorizontalFlip(),
                transforms.RandomResizedCrop(img_size+4, scale=(0.95, 1.))
            ])
            # transform1p = transforms.Compose([
            #     transforms.Resize(size=(img_size+32, img_size+32)), #for pretraining, crop can be bigger since it doesn't matter when bird is not fully visible
            #     TrivialAugmentWideNoColor(),
            #     transforms.RandomHorizontalFlip(),
            #     transforms.RandomResizedCrop(img_size+4, scale=(0.95, 1.))
            # ])
            
            transform2 = transforms.Compose([
                                TrivialAugmentWideNoShape(),
                                transforms.RandomCrop(size=(img_size, img_size)), #includes crop
                                transforms.ToTensor(),
                                normalize
                                ])
            
            
            trn_dset, val_dset, tst_dset, psh_dset, curtaskcla, idxs = get_cub_datasetss(cur_dataset, dc['path'], num_tasks,
                                                                                nc_first_task,
                                                                                validation=validation,
                                                                                trn_transform=trn_transform,
                                                                                tst_transform=tst_transform,
                                                                                transform1=transform1,
                                                                                transform2=transform2,
                                                                                transform_noaug=transform_noaug,
                                                                                class_order=dc['class_order'],
                                                                                repeat_task_0=repeat_task_0)
            

        # apply offsets in case of multiple datasets
        if idx_dataset > 0:
            for tt in range(num_tasks):
                trn_dset[tt].labels = [elem + dataset_offset for elem in trn_dset[tt].labels]
                val_dset[tt].labels = [elem + dataset_offset for elem in val_dset[tt].labels]
                tst_dset[tt].labels = [elem + dataset_offset for elem in tst_dset[tt].labels]
                psh_dset[tt].labels = [elem + dataset_offset for elem in psh_dset[tt].labels]
        dataset_offset = dataset_offset + sum([tc[1] for tc in curtaskcla])

        # reassign class idx for multiple dataset case
        curtaskcla = [(tc[0] + idx_dataset * num_tasks, tc[1]) for tc in curtaskcla]

        # extend final taskcla list
        taskcla.extend(curtaskcla)

        # loaders
        for tt in range(num_tasks):
            print(f"Task {tt}: Training dataset size = {len(trn_dset[tt])}")
            print(f"Task {tt}: Validation dataset size = {len(val_dset[tt])}")
            print(f"Task {tt}: Test dataset size = {len(tst_dset[tt])}")
            if parallelization=="DDP":
                trn_load.append(data.DataLoader(trn_dset[tt], batch_size=batch_size, shuffle=False, num_workers=num_workers,
                                                pin_memory=pin_memory, sampler=DistributedSampler(trn_dset[tt])))
                val_load.append(data.DataLoader(val_dset[tt], batch_size=batch_size, shuffle=False, num_workers=num_workers,
                                                pin_memory=pin_memory, sampler=DistributedSampler(val_dset[tt])))
                tst_load.append(data.DataLoader(tst_dset[tt], batch_size=batch_size, shuffle=False, num_workers=num_workers,
                                                pin_memory=pin_memory, sampler=DistributedSampler(tst_dset[tt])))
                psh_load.append(data.DataLoader(psh_dset[tt], batch_size=batch_size, shuffle=False, num_workers=num_workers,
                                                pin_memory=pin_memory))
            else:
                trn_load.append(data.DataLoader(trn_dset[tt], batch_size=batch_size, shuffle=False, num_workers=num_workers,
                                                pin_memory=pin_memory))
                val_load.append(data.DataLoader(val_dset[tt], batch_size=batch_size, shuffle=False, num_workers=num_workers,
                                                pin_memory=pin_memory))
                tst_load.append(data.DataLoader(tst_dset[tt], batch_size=batch_size, shuffle=False, num_workers=num_workers,
                                                pin_memory=pin_memory))
                psh_load.append(data.DataLoader(psh_dset[tt], batch_size=batch_size, shuffle=False, num_workers=num_workers,
                                                pin_memory=pin_memory))
    return trn_load, val_load, tst_load, psh_load, taskcla, trn_dset, tst_dset, idxs

In [16]:
#Get loaders
trn_loader, _, tst_loader, _, _,trn_dataset, tst_dataset, idxs = get_loaderss(["cub_200_2011_cropped"], 4,
                                                                          None,
                                                                          1, num_workers=4,
                                                                          pin_memory=True,
                                                                          repeat_task_0=False,
                                                                          parallelization="DP")

Processing dataset: cub_200_2011_cropped
{'path': '/raid/home/fdivaler/XAI_CL/ProtoPNet/datasets/cub200_cropped/', 'resize': (224, 224), 'crop': None, 'flip': False, 'online_augment': False, 'normalize': ((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)), 'pad': None, 'class_order': None, 'extend_channel': None}
GET DATA2
Classes per task: [50 50 50 50]
mcfratm
Task 0: Training dataset size = 39300
Task 0: Validation dataset size = 5670
Task 0: Test dataset size = 1452
Task 1: Training dataset size = 39330
Task 1: Validation dataset size = 5610
Task 1: Test dataset size = 1450
Task 2: Training dataset size = 32670
Task 2: Validation dataset size = 12300
Task 2: Test dataset size = 1412
Task 3: Training dataset size = 36600
Task 3: Validation dataset size = 8340
Task 3: Test dataset size = 1480


In [ ]:
# Uncomment to load Stanford Cars dataset
# trn_loader, _, tst_loader, _, _,trn_dataset, tst_dataset, idxs = get_loaderss(["cars"], 7,
#                                                                           None,
#                                                                           1, num_workers=4,
#                                                                           pin_memory=True,
#                                                                           repeat_task_0=False,
#                                                                           use_pipnet=True,
#                                                                           parallelization="DP")

In [17]:
# Compute reverse indices
idxs_reverse = {}
for t in range(num_tasks):
    idxs_reverse[t] = {'trn':{},'tst':{}}
    idxs_reverse[t]['trn'] = {v:k for k,v in idxs[t]['trn'].items()}
    idxs_reverse[t]['tst'] = {v:k for k,v in idxs[t]['tst'].items()}


VISUALIZATION

In [117]:
from tqdm import tqdm
import argparse
import torch
import torch.nn.functional as F
import torch.utils.data
import os
from PIL import Image, ImageDraw as D
import torchvision.transforms as transforms
import torchvision
import random
import matplotlib.pyplot as plt
import shutil

try:
    import cv2
    use_opencv = True
except ImportError:
    use_opencv = False
    print("Heatmaps showing where a prototype is found will not be generated because OpenCV is not installed.", flush=True)


def get_patch_size(img_size, wshape):
    patchsize = 32
    skip = round((img_size - patchsize) / (wshape-1))
    return patchsize, skip

def init_weights_xavier(m):
    if type(m) == torch.nn.Conv2d:
        torch.nn.init.xavier_uniform_(m.weight, gain=torch.nn.init.calculate_gain('sigmoid'))

# https://gist.github.com/weiaicunzai/2a5ae6eac6712c70bde0630f3e76b77b?permalink_comment_id=3662215#gistcomment-3662215
def topk_accuracy(output, target, topk=[1,]):
    """
    Computes the accuracy over the k top predictions for the specified values of k
    """
    with torch.no_grad():
        topk2 = [x for x in topk if x <= output.shape[1]] #ensures that k is not larger than number of classes
        maxk = max(topk2)

        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = (pred == target.unsqueeze(dim=0)).expand_as(pred)

        res = []
        for k in topk:
            if k in topk2:
                correct_k = correct[:k].reshape(-1).float()
                res.append(correct_k)
            else:
                res.append(torch.zeros_like(target))
        return res


In [116]:
img_size = 224
shape = (3, img_size, img_size)
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)
normalize = transforms.Normalize(mean=mean,std=std)
transform_noaug = transforms.Compose([  transforms.Resize(size=(img_size, img_size)),
                                        transforms.ToTensor(),
                                        normalize
                                    ])
#SET PATH TO TRAIN DATASET
# projectset = torchvision.datasets.ImageFolder('/path/to/train/dataset', transform=transform_noaug)
projectset = torchvision.datasets.ImageFolder('/raid/home/fdivaler/XAI_CL/ProtoPNet/datasets/CUB_200_2011/dataset/train', transform=transform_noaug)

#SET PATH TO TEST DATASET
# testset_projection = torchvision.datasets.ImageFolder('/path/to/test/dataset', transform=transform_noaug)
testset_projection = torchvision.datasets.ImageFolder('/raid/home/fdivaler/XAI_CL/ProtoPNet/datasets/CUB_200_2011/dataset/test_full', transform=transform_noaug)

projectloader = torch.utils.data.DataLoader(projectset,
                                            batch_size = 1,
                                            shuffle=False,
                                            pin_memory=True,
                                            num_workers=4,
                                            worker_init_fn=np.random.seed(seed),
                                            drop_last=False
                                            )
test_projectloader = torch.utils.data.DataLoader(testset_projection,
                                             batch_size=1,
                                             shuffle=False,
                                             pin_memory=True,
                                             num_workers=4,
                                             worker_init_fn=np.random.seed(seed),
                                             drop_last=False
                                             )

In [20]:
#Organize indices
united_idxs = {}
united_idxs_rev = {}
for t in range(num_tasks):
    for k in idxs[t].keys():
        for kk,vv in idxs[t][k].items():
            united_idxs[kk] = vv
        for kk1,vv1 in idxs_reverse[t][k].items():
            united_idxs_rev[kk1] = vv1


In [115]:
#Helper function
# convert latent location to coordinates of image patch
def get_img_coordinates(img_size, softmaxes_shape, patchsize, skip, h_idx, w_idx):
    # in case latent output size is 26x26. For convnext with smaller strides. 
    if softmaxes_shape[1] == 26 and softmaxes_shape[2] == 26:
        #Since the outer latent patches have a smaller receptive field, skip size is set to 4 for the first and last patch. 8 for rest.
        h_coor_min = max(0,(h_idx-1)*skip+4)
        if h_idx < softmaxes_shape[-1]-1:
            h_coor_max = h_coor_min + patchsize
        else:
            h_coor_min -= 4
            h_coor_max = h_coor_min + patchsize
        w_coor_min = max(0,(w_idx-1)*skip+4)
        if w_idx < softmaxes_shape[-1]-1:
            w_coor_max = w_coor_min + patchsize
        else:
            w_coor_min -= 4
            w_coor_max = w_coor_min + patchsize
    else:
        h_coor_min = h_idx*skip
        h_coor_max = min(img_size, h_idx*skip+patchsize)
        w_coor_min = w_idx*skip
        w_coor_max = min(img_size, w_idx*skip+patchsize)                                    
    
    if h_idx == softmaxes_shape[1]-1:
        h_coor_max = img_size
    if w_idx == softmaxes_shape[2] -1:
        w_coor_max = img_size
    if h_coor_max == img_size:
        h_coor_min = img_size-patchsize
    if w_coor_max == img_size:
        w_coor_min = img_size-patchsize

    return h_coor_min, h_coor_max, w_coor_min, w_coor_max
    
#Function to visualize the topk images per prototype
@torch.no_grad()                    
def visualize_topk(net, task, projectloader, device, foldername, log_dir, img_size, wshape, k=10):
    print("Visualizing prototypes for topk...", flush=True)
    dir = os.path.join(log_dir, foldername)
    if not os.path.exists(dir):
        os.makedirs(dir)

    near_imgs_dirs = dict()
    seen_max = dict()
    saved = dict()
    saved_ys = dict()
    tensors_per_prototype = dict()
    
    for p in range(net.module._num_prototypes):
        near_imgs_dir = os.path.join(dir, str(p))
        near_imgs_dirs[p]=near_imgs_dir
        seen_max[p]=0.
        saved[p]=0
        saved_ys[p]=[]
        tensors_per_prototype[p]=[]
    
    patchsize, skip = get_patch_size(img_size, wshape)

    imgs = projectloader.dataset.imgs
    
    # Make sure the model is in evaluation mode
    net.eval()
    
    if task is not None: #TASK AWARE
        classification_weights = F.normalize(F.softplus(net.module._classification[task].weight),dim=1)
    else: #TASK AGNOSTIC
        classification_weights = torch.cat([F.normalize(F.softplus(module.weight),dim=1) for module in net.module._classification], dim=0)
    print(classification_weights.shape)

    # Show progress on progress bar
    img_iter = tqdm(enumerate(projectloader),
                    total=len(projectloader),
                    mininterval=50.,
                    desc='Collecting topk',
                    ncols=0)

    # Iterate through the data
    images_seen = 0
    topks = dict()
    # Iterate through the training set
    for i, (xs, ys) in img_iter:
        images_seen+=1
        if task is not None:
            if ysj.item() not in idxs[task]['trn'].values():
                continue
     
        ys = torch.tensor([united_idxs_rev[ys.item()]])
        xs, ys = xs.to(device), ys.to(device)
        

        with torch.no_grad():
            # Use the model to classify this batch of input data
            pfs, pooled, out = net(xs, inference=True, task=task)
            pooled = F.normalize(pooled, dim=0).squeeze(0) 
            pfs = pfs.squeeze(0) 
            imp = classification_weights*pooled
            
            for p in range(pooled.shape[0]):
                c_weight, ci = torch.max(imp[:,p],dim=0) 
                if c_weight > 0.01:#ignore prototypes that are not relevant to any class
                    if p not in topks.keys():
                        topks[p] = []
                        
                    if len(topks[p]) < k:
                        topks[p].append((i, imp[ci,p].item()))
                    else:
                        topks[p] = sorted(topks[p], key=lambda tup: tup[1], reverse=True)
                        if topks[p][-1][1] < imp[ci,p].item():
                            topks[p][-1] = (i, imp[ci,p].item())
                        if topks[p][-1][1] == imp[ci,p].item():
                            # equal scores. randomly chose one (since dataset is not shuffled so latter images with same scores can now also get in topk).
                            replace_choice = random.choice([0, 1])
                            if replace_choice > 0:
                                topks[p][-1] = (i, imp[ci,p].item())
            

    alli = []
    prototypes_not_used = []
    for p in topks.keys():
        found = False
        for idx, score in topks[p]:
            alli.append(idx)
            if score > 0.01:  #in case prototypes have fewer than k well-related patches
                found = True
        if not found:
            prototypes_not_used.append(p)

    print(len(prototypes_not_used), "prototypes do not have any similarity score > 0.01. Will be ignored in visualisation.")
    abstained = 0
    # Show progress on progress bar
    img_iter = tqdm(enumerate(projectloader),
                    total=len(projectloader),
                    mininterval=50.,
                    desc='Visualizing topk',
                    ncols=0)
    for i, (xs, ys) in img_iter: #shuffle is false so should lead to same order as in imgs
        if i in alli:
            ys = torch.tensor([united_idxs_rev[ys.item()]])
            xs, ys = xs.to(device), ys.to(device)
            for p in topks.keys():
                if p not in prototypes_not_used:
                    for idx, score in topks[p]:
                        if idx == i:
                            # Use the model to classify this batch of input data
                            with torch.no_grad():
                                softmaxes, pooled, out = net(xs, inference=True, task=task) #softmaxes has shape (1, num_prototypes, W, H)
                                imp = classification_weights*F.normalize(pooled.squeeze(0),dim=0)
                                outmax = torch.amax(out,dim=1)[0] #shape ([1]) because batch size of projectloader is 1
                                if outmax.item() == 0.:
                                    abstained+=1
                            
                            # Take the max per prototype.                             
                            max_per_prototype, max_idx_per_prototype = torch.max(softmaxes, dim=0)
                            max_per_prototype_h, max_idx_per_prototype_h = torch.max(max_per_prototype, dim=1)
                            max_per_prototype_w, max_idx_per_prototype_w = torch.max(max_per_prototype_h, dim=1) #shape (num_prototypes)
                            
                            c_weight, ci = torch.max(imp[:,p],dim=0) #ignore prototypes that are not relevant to any class
                            #print(c_weight)
                            if (c_weight > 1e-10) or ('pretrain' in foldername):
                                
                                h_idx = max_idx_per_prototype_h[p, max_idx_per_prototype_w[p]]
                                w_idx = max_idx_per_prototype_w[p]
                                
                                # Ensure values are in range [0, 1] for visualization
                                image_numpy1 = (xs - xs.min()) / (xs.max() - xs.min())
                            
                                h_coor_min, h_coor_max, w_coor_min, w_coor_max = get_img_coordinates(img_size, softmaxes.shape, patchsize, skip, h_idx, w_idx)
                                img_tensor_patch = image_numpy1[0, :, h_coor_min:h_coor_max, w_coor_min:w_coor_max].to(device)
                                        
                                saved[p]+=1
                                tensors_per_prototype[p].append(img_tensor_patch.to(device))

    print("Abstained: ", abstained, flush=True)
    all_tensors = []
    for p in range(net.module._num_prototypes):
        if saved[p]>0:
            # add text next to each topk-grid, to easily see which prototype it is
            text = "P "+str(p)
            txtimage = Image.new("RGB", (img_tensor_patch.shape[1],img_tensor_patch.shape[2]), (0, 0, 0))
            draw = D.Draw(txtimage)
            draw.text((img_tensor_patch.shape[0]//2, img_tensor_patch.shape[1]//2), text, anchor='mm', fill="white")
            txttensor = transforms.ToTensor()(txtimage)
            if len(tensors_per_prototype[p])<k:
                for l in range(k-len(tensors_per_prototype[p])):
                    tensors_per_prototype[p].append(torch.zeros_like(tensors_per_prototype[p][-1]).to(device))
            tensors_per_prototype[p].append(txttensor.to(device))
            
            if saved[p]>=k:
                all_tensors+=tensors_per_prototype[p]
            
    if len(all_tensors)>0:
        grid = torchvision.utils.make_grid(all_tensors, nrow=k+1, padding=1)
        torchvision.utils.save_image(grid,os.path.join(dir,"grid_topk_all.png"))
    else:
        print("Pretrained prototypes not visualized. Try to pretrain longer.", flush=True)
    return topks
        


In [ ]:
#VISUALIZE PROTOTYPES
seed_everything(seed=1)
model.to(device)
#Task aware topk
for ti in range(num_tasks):
    topks = visualize_topk(model, task=ti, projectloader=projectloader, device=device, foldername='visualised_prototypes_topk/taw'+str(ti), log_dir=path+'visualize',img_size=224, wshape=26)

#Uncomment for task agnostic topk
#topks = visualize_topk(model, task=None, projectloader=projectloader, device=device, foldername='visualised_prototypes_topk/tag', log_dir=path+'visualize',img_size=224, wshape=26)

Visualizing prototypes for topk...
torch.Size([200, 768])


0 prototypes do not have any similarity score > 0.01. Will be ignored in visualisation.



Visualizing topk: 100% 5995/5995 [11:07<00:00,  8.99it/s] 

Abstained:  0


In [ ]:
#Function for local explanations
def vis_pred(net, task, vis_test_dir, device, log_dir, dir_for_saving_images, num_workers, image_size, wshape, disable_cuda):
    # Make sure the model is in evaluation mode
    net.eval()

    save_dir = os.path.join(log_dir, dir_for_saving_images)
    if os.path.exists(save_dir):
        shutil.rmtree(save_dir)

    patchsize, skip = get_patch_size(image_size, wshape)

    num_workers = num_workers

    mean = (0.485, 0.456, 0.406)
    std = (0.229, 0.224, 0.225)
    normalize = transforms.Normalize(mean=mean,std=std)
    transform_no_augment = transforms.Compose([
                            transforms.Resize(size=(image_size, image_size)),
                            transforms.ToTensor(),
                            normalize])

    vis_test_set = torchvision.datasets.ImageFolder(vis_test_dir, transform=transform_no_augment)
    vis_test_loader = torch.utils.data.DataLoader(vis_test_set, batch_size = 1,
                                                shuffle=False, pin_memory=not disable_cuda and torch.cuda.is_available(),
                                                num_workers=num_workers)
    
    imgs = vis_test_set.imgs
    
    last_y = -1
    new_idxs = idxs.copy()
    if task is not None: #TASK AWARE
        classification_weights = F.normalize(F.softplus(net.module._classification[task].weight),dim=1)
    else: #TASK AGNOSTIC
        classification_weights = torch.cat([F.normalize(F.softplus(module.weight),dim=1) for module in net.module._classification], dim=0)
        l = len(net.module._classification)-1
        for tt in range(l):
            new_idxs[tt]['tst'].update(idxs[tt]['tst']) 
    print(classification_weights.shape)
    if task is not None:
        nidxsval = new_idxs[task]['tst'].values()
    for k, (xs, ys) in enumerate(vis_test_loader): #shuffle is false so should lead to same order as in imgs
        ysit = ys.item()
        if task is not None:
            if ysit not in nidxsval:
                continue
        yso = ys.to(device)
        ys = torch.tensor([united_idxs_rev[ysit]])
        if ys[0] != last_y:
            last_y = ys[0]
            count_per_y = 0
        else:
            count_per_y +=1
            if count_per_y>5: #show max 5 imgs per class to speed up the process
                continue
        xs, ys = xs.to(device), ys.to(device)
        img = imgs[k][0]
        img_name = os.path.splitext(os.path.basename(img))[0]
        dir = os.path.join(save_dir,img_name)
        if not os.path.exists(dir):
            os.makedirs(dir)
            shutil.copy(img, dir)
        image = transforms.Resize(size=(image_size, image_size))(Image.open(img))
        img_tensor = transforms.ToTensor()(image).unsqueeze_(0) #shape (1, 3, h, w)
        with torch.no_grad():
            softmaxes, pooled, out = net(xs, inference=True, task=task) #softmaxes has shape (bs, num_prototypes, W, H), pooled has shape (bs, num_prototypes), out has shape (bs, num_classes)
            # sorted_out, sorted_out_indices = torch.sort(out.squeeze(0), descending=True)
            sorted_out, sorted_out_indices = out.squeeze(0).topk(3, largest=True)

            for pred_class_idx in sorted_out_indices[:3]:
                pred_class_idx_item = pred_class_idx.item()
                if task is not None:
                    pred_class = new_idxs[task]['tst'][pred_class_idx_item+(task*50)] #pred_class è la classe originaria del dataset
                else:
                    pred_class = united_idxs[pred_class_idx_item] #DA VEDERE PER IL TASK AGNOSTIC
                save_path = os.path.join(dir, str(pred_class)+"_"+str(yso.item())+"_"+str(f"{out[0,pred_class_idx].item():.3f}"))
                if not os.path.exists(save_path):
                    os.makedirs(save_path)
                sorted_pooled, sorted_pooled_indices = torch.sort(pooled.squeeze(0), descending=True)
                simweights = []

                simweights_all = pooled[0] * classification_weights[pred_class_idx]
                keep_mask = simweights_all.abs() > 0.01
                prototype_indices_kept = keep_mask.nonzero(as_tuple=False).squeeze(1)
                for prototype_idx in prototype_indices_kept:  
                    simweight = simweights_all[prototype_idx].item()
                    simweights.append(simweight)
                    
                    flat_idx = softmaxes[0, prototype_idx].view(-1).argmax()           # single pass
                    h_size   = softmaxes.size(-2)
                    w_size   = softmaxes.size(-1)
                    max_idx_h = torch.div(flat_idx, w_size, rounding_mode='floor').item()
                    max_idx_w = (flat_idx %  w_size).item()
                    
                    
                    h_coor_min, h_coor_max, w_coor_min, w_coor_max = get_img_coordinates(image_size, softmaxes.shape, patchsize, skip, max_idx_h, max_idx_w)
                    img_tensor_patch = img_tensor[0, :, h_coor_min:h_coor_max, w_coor_min:w_coor_max]
                    image_c = image.copy()
                    draw = D.Draw(image_c)
                    draw.rectangle([(max_idx_w*skip,max_idx_h*skip), (min(image_size, max_idx_w*skip+patchsize), min(image_size, max_idx_h*skip+patchsize))], outline='yellow', width=2)
                    image_c.save(os.path.join(save_path, 'mul%s_p%s_sim%s_w%s_rect.png'%(str(f"{simweight:.3f}"),str(prototype_idx.item()),str(f"{pooled[0,prototype_idx].item():.3f}"),str(f"{classification_weights[pred_class_idx, prototype_idx].item():.3f}"))))

                    # visualise softmaxes as heatmap
                    if use_opencv:
                        heatmap_small = torch.nn.functional.interpolate(
                                            softmaxes[0, prototype_idx].unsqueeze(0).unsqueeze(0),      # (1,1,h,w)
                                            size=image_size,
                                            mode="bilinear",
                                            align_corners=False
                                        ).squeeze().cpu().numpy()
                        heatmap_uint8 = np.uint8(255 * heatmap_small / heatmap_small.max())
                        heatmap_rgb   = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
                        overlay       = 0.4 * heatmap_rgb + 0.6 * (img_tensor.squeeze().permute(1,2,0).cpu().numpy()*255)
                        cv2.imwrite(os.path.join(save_path, f"heatmap_p{prototype_idx.item()}.png"), overlay)
        

In [ ]:
#Local Explanations for predictions
seed_everything(seed=1)
testset_img0_path = test_projectloader.dataset.samples[0][0]
test_path = os.path.split(os.path.split(testset_img0_path)[0])[0]
model.to(device)

# UNCOMMENT FOR TASK AWARE EXPLANATIONS (TIL)
#
# for ti in range(num_tasks):
#     vis_pred(model, task=ti, vis_test_dir=test_path, device=device, log_dir=path+'visualize',
#          dir_for_saving_images='predictions_taw'+str(ti), num_workers=4, image_size=img_size, wshape=26, disable_cuda=False)


# UNCOMMENT FOR TASK AGNOSTIC EXPLANATIONS (CIL)
#
# vis_pred(model, task=None, vis_test_dir=test_path, device=device, log_dir=path+'visualize',
#           dir_for_saving_images='predictions_tag', num_workers=4, image_size=img_size, wshape=26, disable_cuda=False)

torch.Size([200, 768])


/tmp/ipykernel_3945631/1277725945.py:90: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  max_idx_h = (flat_idx // w_size).item()


PROTOTYPE ACTIVATION DRIFT

In [102]:
#Function to visualize eventual drift of regularized prototypes
import pickle
@torch.no_grad()   
def drift(net, task, projectloader, num_classes, device, foldername, log_dir, img_size, wshape):
    checkpoint = checkpoint = torch.load(path+"models/model_task0.pt", map_location=device)
    proto_used = checkpoint['proto_used']
    ridxs = torch.zeros_like(proto_used[0],dtype=torch.bool)
    
    total_elements = proto_used[task].numel()
    total_sum = proto_used[task].sum()
    total_sq_sum = (proto_used[task]**2).sum()

    global_mean = total_sum / total_elements
    global_mean_sq = total_sq_sum / total_elements
    global_var = global_mean_sq - global_mean ** 2
    global_std = global_var.sqrt()

    # Normalize and sum
    norm_t = (proto_used[task] - global_mean) / torch.clamp(global_std, min=1e-4)
    perc = torch.quantile(norm_t,0.75)
    
    ridxs += norm_t >= perc
    idxs2 = torch.nonzero(ridxs, as_tuple=False)
                 
    savedir = os.path.join(log_dir, foldername)
    if not os.path.exists(savedir):
        os.makedirs(savedir)
    
    patchsize, skip = get_patch_size(img_size, wshape)

    imgs = projectloader.dataset.images
    
    # Make sure the model is in evaluation mode
    net.eval()
    
    if task is not None: #TASK AWARE
        classification_weights = F.normalize(F.softplus(net.module._classification[task].weight),dim=1)
    else: #TASK AGNOSTIC
        classification_weights = torch.cat([F.normalize(F.softplus(module.weight),dim=1) for module in net.module._classification], dim=0)
    print(classification_weights.shape)

    # Show progress on progress bar
    img_iter = tqdm(enumerate(projectloader),
                    total=len(projectloader),
                    mininterval=50.,
                    desc='Collecting max prototypes',
                    ncols=0)

    
    max_pooleds = {k:[torch.zeros(classification_weights[0].shape).to(device), torch.zeros(classification_weights[0].shape).to(device)] for k in range(classification_weights.shape[0])}
    # min_pooleds = {k:[None, 0, 1.] for k in range(classification_weights.shape[0])}
    seed_everything(seed=1)
    # Iterate through the data
    images_seen = 0
    imgs_idxs = torch.ones(classification_weights[0].shape, device=device)
    # Iterate through the training set
    for i, (xs, ys) in img_iter:
        images_seen+=1
        # if task is not None:
        #     if ys.item() not in idxs[task]['trn'].values():
        #         continue
        # ys = torch.tensor([united_idxs_rev[ys.item()]])
        xs, ys = xs.to(device), ys.to(device)
        ysit = ys.item()

        with torch.no_grad():
            # Use the model to classify this batch of input data
            _, pooled, _ = net(xs, inference=True, task=task)
            # print(out.shape)
            pooled = pooled.squeeze(0) 
            # pfs = pfs.squeeze(0) 
            crow = classification_weights[ysit]
            sorted_out, sorted_out_indices = torch.sort(crow, descending=True)
            proto_idxs = sorted_out_indices[:idxs2.shape[0]].to(device)
            imp = proto_idxs[torch.isin(proto_idxs,idxs2)].to(device)
            ixs = torch.zeros(classification_weights[ysit].shape, dtype=torch.bool)
            ixs[imp] = True
            tmp = classification_weights[ysit]*F.normalize(pooled,dim=0)
            tmp_ixs = tmp*ixs.to(device)
            

            # Extract the relevant values
            current_values = max_pooleds[ysit][1][imp].to(device)
            new_values = tmp_ixs[imp].to(device)
            # Create a mask where new_values > current_values
            mask = new_values > current_values
            # Update only where the condition is true
            if mask.sum() == 0:
                continue
            max_pooleds[ysit][1][imp[mask]] = new_values[mask].to(device)
            max_pooleds[ysit][0][imp[mask]] = (imgs_idxs[imp[mask]]*i).to(device)
    

    
    img_idxs_max = []
    for k, v in max_pooleds.items():
        ll, im = torch.sort(max_pooleds[k][1], descending=True)
        # img_idxs = sorted([int(e.item()) for e in max_pooleds[k][0][ip]])
        img_idxs_max.append(int(max_pooleds[k][0][im[0]].item()))
    
    activations_grid_rec = {k:{} for k in range(num_classes)}
    activations_grid = {k:{} for k in range(num_classes)}

    for t in range(task,4):
        seed_everything(seed=1)
        activations = {}
        checkpoint = torch.load(path+"models/model_task"+str(t)+".pt", map_location=device)
        net = construct_CIPNet(50, "convnext_tiny_26",True, False)
        net = wrapper(net, devices = [int(device[-1])], parallelization="DP")
        for l in range(1,t+1):
            net.module.add_head(bias=False)

        net.module._net.load_state_dict(checkpoint['backbone'])
        net.module._classification.load_state_dict(checkpoint['classifiers'])
        if 'tau' in checkpoint.keys():
            net.module.tau = checkpoint['tau']

        proto_used = checkpoint['proto_used']
        for k,v in proto_used.items():
            proto_used[k] = v.cpu()
        proto_idxs = checkpoint['proto_idxs'].cpu()

        net.to(device)
        net.eval()
        print("MODELLO CARICATO")

        # Show progress on progress bar
        img_iter = tqdm(enumerate(img_idxs_max),
                    total=len(img_idxs_max),
                    mininterval=20.,
                    desc='Visualizing topk',
                    ncols=0)
        sp = "MAX"
        
        for idx, i in img_iter: #shuffle is false so should lead to same order as in imgs
            xs, ys = projectloader.dataset[i][0].unsqueeze(dim=0), projectloader.dataset[i][1]
            # if task is not None:
            #     if ys not in idxs[task]['trn'].values():
            #         continue

            # ys = torch.tensor([united_idxs_rev[ys]])

            ysit = ys.item()
            
            mm, ps = torch.sort(max_pooleds[ysit][1],descending=True)
            p=ps[0].item()

            img = imgs[i]
            # print(img)
            img_name = os.path.splitext(os.path.basename(img))[0]
            dir = os.path.join(savedir,"class_"+str(ysit)+"_"+str(united_idxs[ysit]))
            if not os.path.exists(dir):
                os.makedirs(dir)
                shutil.copy(img, dir)

            with torch.no_grad():
                softmaxes, pooled, out = net(xs, inference=True, task=task) #softmaxes has shape (bs, num_prototypes, W, H), pooled has shape (bs, num_prototypes), out has shape (bs, num_classes)
                sorted_out, pred_class_idx = torch.max(out.squeeze(0), dim=0)

                activations[i] = softmaxes.cpu()
                
                if t > 0: 
                    with open(path+'visualize/activations_t'+str(t-1)+'.pkl', 'rb') as f:
                        old_activations = pickle.load(f)
                    diff_rec = (old_activations[i][0,p,:,:].to(device) - softmaxes[0,p,:,:])
                    diff_rec = diff_rec.view(diff_rec.size(0),diff_rec.size(1), -1)
                    norm_diff_rec = diff_rec.norm(2, dim=2)
                    activations_grid_rec[ysit][t] = norm_diff_rec.mean()

                    with open(path+'visualize/activations_t'+str(t-t)+'.pkl', 'rb') as f:
                        old_activations = pickle.load(f)
                    diff = (old_activations[i][0,p,:,:].to(device) - softmaxes[0,p,:,:])
                    diff = diff.view(diff.size(0),diff.size(1), -1)
                    norm_diff = diff.norm(2, dim=2)
                    activations_grid[ysit][t] = norm_diff.mean()
                    
                # Convert tensor to numpy and rearrange dimensions to [H, W, C]
                image_numpy1 = xs[0].permute(1, 2, 0).numpy()

                # Ensure values are in range [0, 1] for visualization
                image_numpy1 = (image_numpy1 - image_numpy1.min()) / (image_numpy1.max() - image_numpy1.min())
                # plt.imsave(fname=os.path.join(dir, img_name+'.png'), arr=image_numpy1)

                # visualise softmaxes as heatmap
                if use_opencv:
                    heatmap_small = torch.nn.functional.interpolate(
                                        softmaxes[0, p].unsqueeze(0).unsqueeze(0),      # (1,1,h,w)
                                        size=img_size,
                                        mode="bilinear",
                                        align_corners=False
                                    ).squeeze()
                    
                    heatmap_t = heatmap_small.clamp_min(0)                 
                    heatmap_t = heatmap_t / (heatmap_t.max() + 1e-6)       
                    heatmap_u8 = (heatmap_t * 255).byte().cpu().numpy()    

                    heatmap_bgr  = cv2.applyColorMap(heatmap_u8, cv2.COLORMAP_JET)       
                    base_img_bgr = (image_numpy1 * 255).astype(np.uint8)[..., ::-1]      
                    overlay_bgr  = cv2.addWeighted(heatmap_bgr, 0.4, base_img_bgr, 0.6, 0)
                    cv2.imwrite(
                        os.path.join(
                            dir,
                            f"heatmap_{t}_{sp}_p{p}_{pred_class_idx.item()}_{out[0, pred_class_idx]:.3f}.png"
                        ),
                        overlay_bgr,
                    )

        with open(path+'visualize/activations_t'+str(t)+'.pkl', 'wb') as f:
            pickle.dump(activations, f)
                    

    return savedir, activations_grid_rec, activations_grid


                    
    return savedir


In [104]:
seed_everything(seed=1)
model.to(device)
dir, act_rec, act = drift(model, 0, tst_loader[0], 50, device, 'regularize_protos', log_dir=path+'visualize',img_size=224, wshape=26)
#Images are saved in dir

torch.Size([50, 768])


Using CONVNEXT_TINY_26 as base architecture
Number of prototypes:  768
NUM PROTOTYPES 768
MODELLO CARICATO


Visualizing topk: 100% 50/50 [00:01<00:00, 41.25it/s]


Using CONVNEXT_TINY_26 as base architecture
Number of prototypes:  768
NUM PROTOTYPES 768
CIPNET CLS HEAD ADDED AND INITIALIZED
MODELLO CARICATO


Visualizing topk: 100% 50/50 [00:06<00:00,  7.40it/s]


Using CONVNEXT_TINY_26 as base architecture
Number of prototypes:  768
NUM PROTOTYPES 768
CIPNET CLS HEAD ADDED AND INITIALIZED
CIPNET CLS HEAD ADDED AND INITIALIZED
MODELLO CARICATO


Visualizing topk: 100% 50/50 [00:06<00:00,  7.82it/s]


Using CONVNEXT_TINY_26 as base architecture
Number of prototypes:  768
NUM PROTOTYPES 768
CIPNET CLS HEAD ADDED AND INITIALIZED
CIPNET CLS HEAD ADDED AND INITIALIZED
CIPNET CLS HEAD ADDED AND INITIALIZED
MODELLO CARICATO


Visualizing topk: 100% 50/50 [00:06<00:00,  7.88it/s]


In [105]:
import pandas as pd
# Convert nested dict of tensors to nested dict of floats
float_data = {
    outer_k: {inner_k: v.item() for inner_k, v in inner_dict.items()}
    for outer_k, inner_dict in act.items()
}

# Create DataFrame
df = pd.DataFrame.from_dict(float_data, orient='index')
# print(df)

# Convert nested dict of tensors to nested dict of floats
float_data_rec = {
    outer_k: {inner_k: v.item() for inner_k, v in inner_dict.items()}
    for outer_k, inner_dict in act_rec.items()
}

# Create DataFrame
df_rec = pd.DataFrame.from_dict(float_data_rec, orient='index')
# print(df)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 1, figsize=(3, 2))  # 1 row, 2 columns

# Font sizes
title_fontsize = 17
label_fontsize = 15
tick_fontsize = 13

# X-axis labels (assuming 3 tasks originally labeled 0, 1, 2)
x_ticks = list(range(df.shape[1]))
x_labels = [str(i + 2) for i in x_ticks]  # shift labels to 2, 3, 4

# --- Plot 1 ---
mean_series_1 = df.mean(axis=0)
axes.plot(x_ticks, mean_series_1.values, marker='o')
# axes[0].set_title("Activations Difference Mean to Task t = 1", fontsize=title_fontsize)
axes.set_xlabel("Tasks", fontsize=label_fontsize)
axes.set_ylabel("Mean Value", fontsize=label_fontsize)
axes.grid(True)
axes.set_xticks(x_ticks)
axes.set_xticklabels(x_labels, fontsize=tick_fontsize)
axes.tick_params(axis='y', labelsize=tick_fontsize)
axes.set_ylim(mean_series_1.min() * 0.95, mean_series_1.max() * 1.05)
from matplotlib.ticker import ScalarFormatter

class ScalarFormatterWithDecimals(ScalarFormatter):
    def _set_format(self):
        self.format = '%1.1f'  # Forces one decimal in mantissa

formatter = ScalarFormatterWithDecimals(useMathText=True)
formatter.set_scientific(True)
formatter.set_powerlimits((0, 0))  # Always use scientific notation
axes.yaxis.set_major_formatter(formatter)


plt.tight_layout()
plt.savefig("activations_plots_0.png", dpi=300, bbox_inches='tight')
plt.show()


fig, axes = plt.subplots(1, 1, figsize=(3, 2))
x_ticks = list(range(df.shape[1]))
x_labels = [str(i + 2) for i in x_ticks]  # shift labels to 2, 3, 4
title_fontsize = 17
label_fontsize = 15
tick_fontsize = 13
# --- Plot 2 ---
mean_series_2 = df_rec.mean(axis=0)
axes.plot(x_ticks, mean_series_2.values, marker='o')
# axes[1].set_title("Activations Difference Mean to Previous Tasks", fontsize=title_fontsize)
axes.set_xlabel("Tasks", fontsize=label_fontsize)
axes.set_ylabel("Mean Value", fontsize=label_fontsize)
axes.grid(True)
axes.set_xticks(x_ticks)
axes.set_xticklabels(x_labels, fontsize=tick_fontsize)
axes.tick_params(axis='y', labelsize=tick_fontsize)
axes.set_ylim(mean_series_2.min() * 0.95, mean_series_2.max() * 1.05)
from matplotlib.ticker import ScalarFormatter

class ScalarFormatterWithDecimals(ScalarFormatter):
    def _set_format(self):
        self.format = '%1.1f'  # Forces one decimal in mantissa

formatter = ScalarFormatterWithDecimals(useMathText=True)
formatter.set_scientific(True)
formatter.set_powerlimits((0, 0))  # Always use scientific notation
axes.yaxis.set_major_formatter(formatter)

# Layout adjustment

plt.tight_layout()
plt.savefig("activations_plots_1.png", dpi=300, bbox_inches='tight')
plt.show()
